In [2]:
import pandas as pd
from pathlib import Path

LOG_FILE = Path(r"D:\Downloads\Capstone\Project\logOutput_v2\hdfs_v2_merged_logs.csv")

sample_df = pd.read_csv(
    LOG_FILE,
    usecols=["merged_log"],
    nrows=5
)

print("Columns used:", sample_df.columns.tolist())
print(sample_df.head())
print("File size (MB):", round(LOG_FILE.stat().st_size / (1024 * 1024), 2))

Columns used: ['merged_log']
                                          merged_log
0  2015-12-03 14:37:47,611 INFO org.apache.hadoop...
1  2015-12-03 14:37:47,618 INFO org.apache.hadoop...
2  2015-12-03 14:37:48,253 INFO org.apache.hadoop...
3  2015-12-03 14:37:48,315 INFO org.apache.hadoop...
4  2015-12-03 14:37:48,315 INFO org.apache.hadoop...
File size (MB): 18843.85


In [3]:
import re
import pandas as pd

CHUNK_SIZE = 20000

ip_pattern = re.compile(r"\b(?:\d{1,3}\.){3}\d{1,3}\b")
num_pattern = re.compile(r"\b\d+\b")

def clean_log_text(text):
    text = str(text).lower()
    text = ip_pattern.sub(" IP ", text)
    text = num_pattern.sub(" NUM ", text)
    text = re.sub(r"[^a-z_ ]+", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

chunk_iter = pd.read_csv(
    LOG_FILE,
    usecols=["merged_log"],
    dtype={"merged_log": "string"},
    chunksize=CHUNK_SIZE,
    low_memory=True
)

first_chunk = next(chunk_iter)

preview_df = first_chunk.head(10).copy()
preview_df["clean_text"] = preview_df["merged_log"].map(clean_log_text)

print("Chunk shape:", first_chunk.shape)
print(preview_df[["merged_log", "clean_text"]].head(5))

Chunk shape: (20000, 1)
                                          merged_log  \
0  2015-12-03 14:37:47,611 INFO org.apache.hadoop...   
1  2015-12-03 14:37:47,618 INFO org.apache.hadoop...   
2  2015-12-03 14:37:48,253 INFO org.apache.hadoop...   
3  2015-12-03 14:37:48,315 INFO org.apache.hadoop...   
4  2015-12-03 14:37:48,315 INFO org.apache.hadoop...   

                                          clean_text  
0  info org apache hadoop hdfs server datanode da...  
1  info org apache hadoop hdfs server datanode da...  
2  info org apache hadoop metrics impl metricscon...  
3  info org apache hadoop metrics impl metricssys...  
4  info org apache hadoop metrics impl metricssys...  


In [4]:
import gc

MAX_SAMPLES = 120000   # safe starting size
CHUNK_SIZE = 20000

sample_texts = []
kept = 0

chunk_iter = pd.read_csv(
    LOG_FILE,
    usecols=["merged_log"],
    dtype={"merged_log": "string"},
    chunksize=CHUNK_SIZE,
    low_memory=True
)

for i, chunk in enumerate(chunk_iter, start=1):
    chunk = chunk.dropna(subset=["merged_log"]).copy()
    chunk["clean_text"] = chunk["merged_log"].map(clean_log_text)

    chunk = chunk[chunk["clean_text"].str.len() > 20]

    remaining = MAX_SAMPLES - kept
    if remaining <= 0:
        break

    take_n = min(len(chunk), remaining)
    sample_texts.extend(chunk["clean_text"].iloc[:take_n].tolist())
    kept += take_n

    print(f"Processed chunk {i}, kept so far: {kept}")

    del chunk
    gc.collect()

print("\nTotal sampled logs:", len(sample_texts))
print("\nFirst 3 cleaned samples:")
for t in sample_texts[:3]:
    print("-", t[:200])

Processed chunk 1, kept so far: 20000
Processed chunk 2, kept so far: 40000
Processed chunk 3, kept so far: 60000
Processed chunk 4, kept so far: 80000
Processed chunk 5, kept so far: 100000
Processed chunk 6, kept so far: 120000

Total sampled logs: 120000

First 3 cleaned samples:
- info org apache hadoop hdfs server datanode datanode startup_msg startup_msg starting datanode startup_msg host mesos master startup_msg args startup_msg version startup_msg classpath usr local hadoop
- info org apache hadoop hdfs server datanode datanode registered unix signal handlers for term hup int
- info org apache hadoop metrics impl metricsconfig loaded properties from hadoop metrics properties


In [5]:
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(
    max_features=20000,
    min_df=3,
    max_df=0.95,
    ngram_range=(1, 2),
    dtype=np.float32
)

X_tfidf = tfidf.fit_transform(sample_texts)

print("TF-IDF shape:", X_tfidf.shape)
print("Vocabulary size:", len(tfidf.get_feature_names_out()))

TF-IDF shape: (120000, 1371)
Vocabulary size: 1371


In [6]:
from sklearn.ensemble import IsolationForest
import numpy as np

iso = IsolationForest(
    n_estimators=100,
    contamination=0.05,   # starting assumption: 5% anomalies
    random_state=42,
    n_jobs=-1
)

iso.fit(X_tfidf)

pred_raw = iso.predict(X_tfidf)   # 1 = normal, -1 = anomaly
anomaly_pred = np.where(pred_raw == -1, 1, 0)

anomaly_score = -iso.score_samples(X_tfidf)  # higher = more anomalous

print("Training completed.")
print("Predicted anomaly counts:")
print("Normal :", (anomaly_pred == 0).sum())
print("Anomaly:", (anomaly_pred == 1).sum())

print("\nAnomaly score summary:")
print(pd.Series(anomaly_score).describe())

Training completed.
Predicted anomaly counts:
Normal : 114535
Anomaly: 5465

Anomaly score summary:
count    120000.000000
mean          0.425334
std           0.047024
min           0.365046
25%           0.381998
50%           0.426071
75%           0.439886
max           0.688021
dtype: float64


In [7]:
result_df = pd.DataFrame({
    "text": sample_texts,
    "anomaly_pred": anomaly_pred,
    "anomaly_score": anomaly_score
})

top_anomalies = result_df.sort_values("anomaly_score", ascending=False).head(20)

print(top_anomalies[["anomaly_score", "anomaly_pred"]])

for i, row in enumerate(top_anomalies.itertuples(index=False), start=1):
    print(f"\n--- Top anomaly {i} ---")
    print("Score:", row.anomaly_score)
    print("Pred :", row.anomaly_pred)
    print("Text :", row.text[:500])

        anomaly_score  anomaly_pred
69060        0.688021             1
98815        0.687446             1
493          0.686052             1
67           0.686049             1
73507        0.686049             1
22041        0.686049             1
5926         0.686049             1
114042       0.686049             1
42992        0.686049             1
42980        0.686049             1
643          0.686049             1
73268        0.686049             1
754          0.686049             1
739          0.686049             1
760          0.686049             1
42995        0.686049             1
826          0.686049             1
33060        0.686049             1
862          0.686049             1
598          0.686049             1

--- Top anomaly 1 ---
Score: 0.6880207816046056
Pred : 1
Text : info org apache hadoop hdfs server datanode datanode successfully sent block report x e eb bbcd e containing storage report s of which we sent the reports had total blocks and use